In [3]:
import os
BASE_DIR = "\\notebooks\\experiments\\params_prediction"
if os.getcwd().endswith(BASE_DIR):
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

In [4]:
from src.models.model_handler import save_model, load_model, get_class_def_from_notebook
from src.models.testing_utils import evaluate_continuous, predict_continuous
from src.data.DatasetGenerator import DatasetGenerator
from src.data.preprocessing import min_max_normalize
from src.models.training_utils import train_model
from src.utils.helpers import load_param
from src.constants import params

from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import torch

In [5]:
# Aconseguim el dataset
num_particules = 10000
dg = DatasetGenerator(num_particules)
data = dg.generate_FractionalBrownianMotion(random_params=True, num_passos=200)

c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\fbm\fbm.py:171: UserWarning: Combination of increments n and Hurst value H invalid for Davies-Harte method. Reverting to Hosking method. Occurs when n is small and Hurst is close to 1. 
  warnings.warn(


In [6]:
# Aconseguim un dataframe amb les coordenades i la hurst
df = data[["coords", "attributes"]].copy()
df["hurst"] = df["attributes"].apply(lambda x: x["hurst"])
df = df[["coords", "hurst"]]

In [7]:
# Tractem les coordenades per aconseguir X_tensor
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1) # Per aconseguir la forma (n_samples, 2, seq_length)

In [8]:
# Tractem les coordenades per aconseguir y_tensor
hurst_series = df["hurst"].apply(lambda x: min_max_normalize(x, params.fbm.min_hurst, params.fbm.max_hurst))
labels = hurst_series.values  # Suponiendo que 'hurst' contiene los valores reales
y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

In [9]:
# Aconseguir X_train, X_val, X_test, y_train, y_val i y_test
n, train_end, val_end = len(df), int(0.7 * len(df)), int(0.85 * len(df))
X_train, X_val, X_test = X_tensor[:train_end], X_tensor[train_end:val_end], X_tensor[val_end:]
y_train, y_val, y_test = y_tensor[:train_end], y_tensor[train_end:val_end], y_tensor[val_end:]

In [10]:
# Muntem els Data_loaders
learning_batch_size = 32
try_batch_size = 1

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=learning_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=try_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=try_batch_size, shuffle=True)

In [ ]:
class FractionalBrownianMotionHurstRegressor(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        # Bloque 1
        self.first_conv = nn.Conv1d(input_channels, 32, 3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, 3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, 3, padding=1)
        
        # Bloque 2
        self.conv_32_64 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, 3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, 3, padding=1)
        
        # Bloque 3
        self.conv_64_128 = nn.Conv1d(64, 128, 3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, 3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, 3, padding=1)
        
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool1d(4)  # Corregido a 1D
        self.fc1 = nn.Linear(128*4, 256)         # Ajustada la entrada
        self.fc2 = nn.Linear(256, 1)
        
        # Opcional: Añadir BatchNorm
        self.bn32 = nn.BatchNorm1d(32)
        self.bn64 = nn.BatchNorm1d(64)
        self.bn128 = nn.BatchNorm1d(128)

        self.dropout_conv = nn.Dropout(p=0.1)   # Després de blocs convolucionals
        self.dropout_fc = nn.Dropout(p=0.5)     # Abans de la capa de sortida

    def forward(self, x):
        # Bloque 1
        print("Input shape:", x.shape)
        a_resnet = F.relu(self.first_conv(x))
        a = F.relu(self.conv1_32(a_resnet))
        a = F.relu(self.conv2_32(a))
        print("Input shape:", a.shape)
        a = a + a_resnet
        print("Input shape:", a.shape)
        a = self.dropout_conv(a)  # Dropout després del bloc residual
        print("Input shape:", a.shape)
        a = self.max_pool(a)
        print("Input shape:", a.shape)
        raise ValueError("Debugging shape: " + str(a.shape))
        # Bloque 2
        b_resnet = F.relu(self.conv_32_64(a))
        b = F.relu(self.conv1_64(b_resnet))
        b = F.relu(self.conv2_64(b))
        b = b + b_resnet
        b = self.dropout_conv(b)
        b = self.max_pool(b)

        # Bloque 3
        c_resnet = F.relu(self.conv_64_128(b))
        c = F.relu(self.conv1_128(c_resnet))
        c = F.relu(self.conv2_128(c))
        c = c + c_resnet
        c = self.dropout_conv(c)
        c = self.avg_pool(c)

        # Clasificación
        c = c.view(c.size(0), -1)
        c = F.relu(self.fc1(c))
        c = self.dropout_fc(c)  # Dropout abans de la capa de sortida
        c = self.fc2(c)
        c = torch.tanh(c)
        out = (c + 1) / 2
        
        return out

input_channels = 2

# Inicialitzem el model d'aquesta manera per poder després guardar facilment els paràmetres
model_params = {
    "input_channels": input_channels
}

model = FractionalBrownianMotionHurstRegressor(**model_params)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3),
    mode='min',
    factor=0.1,
    patience=2,
    # verbose=True <--- Deprecated
)

best_model = train_model(
    model=model,
    criterion=nn.MSELoss(),
    optimizer=scheduler.optimizer,
    num_epochs=15,
    train_loader=train_loader,
    val_loader=val_loader,
    scheduler=scheduler,
    print_epoch=True,
    patience=5
)

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "_zmq.py", line 169, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [14]:
evaluate_continuous(best_model, X_test, y_test)

Test MSE: 0.0210


0.021040914580225945

In [15]:
predict_continuous(best_model, X_test, y_test, num_samples=10)

Predicciones:
Ejemplo 1: Predicho = 0.1994394063949585, Real = 0.1492568850517273
Ejemplo 2: Predicho = 0.8731732964515686, Real = 0.8264139890670776
Ejemplo 3: Predicho = 0.6960595846176147, Real = 0.4402088224887848
Ejemplo 4: Predicho = 0.7282288074493408, Real = 0.45365819334983826
Ejemplo 5: Predicho = 0.8276792764663696, Real = 0.6541694402694702
Ejemplo 6: Predicho = 0.879085123538971, Real = 0.8062137365341187
Ejemplo 7: Predicho = 0.7650904655456543, Real = 0.5008019804954529
Ejemplo 8: Predicho = 0.08455148339271545, Real = 0.016350965946912766
Ejemplo 9: Predicho = 0.8712145090103149, Real = 0.8509240746498108
Ejemplo 10: Predicho = 0.46219658851623535, Real = 0.3297498822212219


,Predicción,Valor real
0,0.199439,0.149257
1,0.873173,0.826414
2,0.696060,0.440209
3,0.728229,0.453658
4,0.827679,0.654169
5,0.879085,0.806214
6,0.765090,0.500802
7,0.084551,0.016351
8,0.871215,0.850924
9,0.462197,0.329750


In [16]:
SAVE_PATH = "../../../models/trained_models/"

In [18]:
save_model(model_instance=best_model,
           model_source=get_class_def_from_notebook(best_model.__class__.__name__),
           attributes=model_params,
           input_channels=input_channels,
           path=load_param("path", "model")
           )

In [2]:
prova_carrega_model=load_model(
    load_param("path", "model>trained") +
    best_model.__class__.__name__ +
    ".pth"
)

evaluate_continuous(prova_carrega_model, X_test, y_test)
predict_continuous(prova_carrega_model, X_test, y_test, num_samples=10)

NameError: name 'load_model' is not defined